# Saved MLP Optimizer 测试 Notebook：测试集构造、剔除训练集、加载网络、计算测试指标

这个 notebook 是把当前的 `test_saved_mlp_residual_loss_detailed.py` 拆开重写成可逐步检查的版本。它对应你上传的测试脚本：脚本会读取保存好的 `optimization_report.json` 和 `mlp_optimizer_state_dict.pt`，构造 held-out 测试集，评估测试集 residual / energy loss，并额外画 Adam `lr=1e-4` 的详细收敛轨迹。

你现在怀疑的问题是：**全测试集结果和 `p_n` 单点结果几乎看不出区别**。这个 notebook 增加了几个诊断单元，专门检查这是代码问题，还是测试集设计导致的自然现象。

核心诊断点：

1. 当前测试集是在 `y*` 附近半径 `0.01` 的 cube 中采样；
2. 物理初值 `p_n` 本身也可能落在这个 cube 内，所以它未必是外推点；
3. 这个单帧自由落体问题的能量是严格二次函数，因此 residual 和 loss gap 有确定关系：

$$
E(y)-E(y^*)=\frac{dt^2}{2m}\lVert \nabla E(y) \rVert_2^2.
$$

所以如果模型在这个局部区域表现比较均匀，测试集平均指标和 `p_n` 单点指标接近并不一定是 bug。


## 0. 配置

把 `RESULTS_DIR` 改成你的训练输出目录。该目录应包含：

```text
dataset_scale_ablation_summary.json
<experiment_name>/optimization_report.json
<experiment_name>/mlp_optimizer_state_dict.pt
```


In [5]:
from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Sequence

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# ========= 用户配置 =========
RESULTS_DIR = Path("/data/zhoucy/sim_newton/unit_test_for_scale_data/second_stage_test/free_fall_regular_grid_fullbatch_50000_float64")
                                       # TODO: 改成你的训练输出目录
OUTPUT_DIR = None                      # None -> RESULTS_DIR / "heldout_test_residual_loss_notebook"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

NUM_TEST = 4096
TEST_SEED = 20260617
STEPS = 50
BATCH_SIZE = 8192
RADIUS_SCALE = 1.0

# detailed trajectories: Adam lr=1e-4, N=8 和 N≈10000（实际通常为 10648）
DO_DETAILED_TRAJECTORIES = True
DETAILED_OPTIMIZER_NAME = "adam"
DETAILED_LEARNING_RATE = 1e-4
DETAILED_TARGET_DATASET_SIZES = [8, 10_000]
NUM_DETAILED_TEST_POINTS = 3

PLOT_FLOOR = 1e-14
GRID_MATCH_TOL = 1e-10
MAX_TRAINING_POINTS_PER_SUBPLOT = 6000
MAX_TEST_POINTS_FOR_DISTRIBUTION = 6000

RESULTS_DIR = RESULTS_DIR.expanduser().resolve()
OUTPUT_DIR = RESULTS_DIR / "heldout_test_residual_loss_notebook" if OUTPUT_DIR is None else Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device(DEVICE)

print("RESULTS_DIR:", RESULTS_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("DEVICE     :", device)

RESULTS_DIR: /data/zhoucy/sim_newton/unit_test_for_scale_data/second_stage_test/free_fall_regular_grid_fullbatch_50000_float64
OUTPUT_DIR : /data/zhoucy/sim_newton/unit_test_for_scale_data/second_stage_test/free_fall_regular_grid_fullbatch_50000_float64/heldout_test_residual_loss_notebook
DEVICE     : cuda:0


## 1. 物理问题：能量、残差、Newton 更新

单帧自由落体的隐式欧拉变分能量是：

$$
E(y)=\frac{m}{2dt^2}\lVert y-p_n-dt v_n \rVert_2^2 + mg y_z.
$$

驻点残差就是梯度：

$$
\nabla E(y)=\frac{m}{dt^2}(y-p_n-dt v_n)+(0,0,mg).
$$

解析最优解是：

$$
y^*=p_n+dt v_n-dt^2(0,0,g).
$$

因为 Hessian 是常数矩阵：

$$
\nabla^2E(y)=\frac{m}{dt^2}I,
$$

所以能量 gap 和 residual norm 满足：

$$
E(y)-E(y^*)=\frac{dt^2}{2m}\lVert \nabla E(y) \rVert_2^2.
$$

这说明 residual 和 loss gap 不是完全独立的两个指标，而是平方关系。


In [6]:
def variational_energy(y: torch.Tensor, p_n: torch.Tensor, v_n: torch.Tensor, m: float, g: float, dt: float) -> torch.Tensor:
    residual = y - p_n - dt * v_n
    kinetic_term = (m / (2.0 * dt**2)) * torch.sum(residual**2, dim=-1)
    potential_term = m * g * y[..., 2]
    return kinetic_term + potential_term


def stationarity_residual(y: torch.Tensor, p_n: torch.Tensor, v_n: torch.Tensor, m: float, g: float, dt: float) -> torch.Tensor:
    residual = (m / dt**2) * (y - p_n - dt * v_n)
    gravity = torch.zeros_like(residual)
    gravity[..., 2] = m * g
    return residual + gravity


def stationarity_residual_norm(y: torch.Tensor, p_n: torch.Tensor, v_n: torch.Tensor, m: float, g: float, dt: float) -> torch.Tensor:
    return torch.linalg.vector_norm(stationarity_residual(y, p_n, v_n, m, g, dt), dim=-1)


def newton_direction(y: torch.Tensor, p_n: torch.Tensor, v_n: torch.Tensor, m: float, g: float, dt: float) -> torch.Tensor:
    grad = stationarity_residual(y, p_n, v_n, m, g, dt)
    return -(dt**2 / m) * grad


def finite_plot_value(value: float | int | None) -> float:
    if value is None:
        return float("nan")
    value = float(value)
    if not math.isfinite(value):
        return float("nan")
    return max(value, PLOT_FLOOR)

## 2. 网络结构：必须和训练脚本一致

训练脚本保存的是 `state_dict`，里面不只有 Linear 层参数，也包括 `input_mean` 和 `input_std` 这两个 buffer。测试时必须按同样结构初始化网络，并加载同一个 `state_dict`。


In [7]:
class MLPOptimizer(nn.Module):
    """训练脚本中的 12 -> 32 -> 32 -> 3 学习型迭代优化器。"""

    def __init__(self, *, dtype: torch.dtype, use_input_normalization: bool = True, use_output_dt_scaling: bool = True,
                 input_mean: torch.Tensor | None = None, input_std: torch.Tensor | None = None) -> None:
        super().__init__()
        self.use_input_normalization = bool(use_input_normalization)
        self.use_output_dt_scaling = bool(use_output_dt_scaling)
        self.net = nn.Sequential(
            nn.Linear(12, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 3)
        )
        if input_mean is None:
            input_mean = torch.zeros(12, dtype=dtype)
        if input_std is None:
            input_std = torch.ones(12, dtype=dtype)
        self.register_buffer("input_mean", input_mean.clone().detach().to(dtype=dtype))
        self.register_buffer("input_std", input_std.clone().detach().to(dtype=dtype))
        self.to(dtype=dtype)

    @staticmethod
    def _expand_feature_for_batch(feature: torch.Tensor, batch_size: int) -> torch.Tensor:
        if feature.ndim == 1:
            return feature.unsqueeze(0).expand(batch_size, -1)
        if feature.ndim == 2 and feature.shape[0] == batch_size:
            return feature
        raise ValueError(f"Feature shape is incompatible: feature={tuple(feature.shape)}, batch={batch_size}")

    def forward(self, y: torch.Tensor, history: torch.Tensor, params: torch.Tensor) -> torch.Tensor:
        if y.ndim == 1:
            inp = torch.cat([y, history, params], dim=-1)
            if self.use_input_normalization:
                inp = (inp - self.input_mean) / self.input_std
            delta = self.net(inp)
            if self.use_output_dt_scaling:
                delta = params[2] * delta
            return delta

        if y.ndim != 2 or y.shape[-1] != 3:
            raise ValueError(f"Expected y shape [3] or [B, 3], got {tuple(y.shape)}")
        batch_size = y.shape[0]
        history_batch = self._expand_feature_for_batch(history, batch_size)
        params_batch = self._expand_feature_for_batch(params, batch_size)
        inp = torch.cat([y, history_batch, params_batch], dim=-1)
        if self.use_input_normalization:
            inp = (inp - self.input_mean) / self.input_std
        delta = self.net(inp)
        if self.use_output_dt_scaling:
            delta = params_batch[:, 2:3] * delta
        return delta

## 3. 扫描训练结果目录

每个实验子目录应包含：

- `optimization_report.json`
- `mlp_optimizer_state_dict.pt`

从 `optimization_report.json` 中读取训练集规模、优化器参数、物理参数和归一化参数。


In [8]:
@dataclass(frozen=True)
class ExperimentFile:
    experiment_dir: Path
    report_path: Path
    model_path: Path
    report: dict[str, Any]


@dataclass(frozen=True)
class RegularGridSpec:
    target_dataset_size: int
    actual_dataset_size: int
    points_per_axis: int
    sampling_radius: float
    axis_spacing: float


def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def safe_torch_load_state_dict(path: Path) -> dict[str, torch.Tensor]:
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    if not isinstance(state, dict):
        raise TypeError(f"{path} does not contain a state_dict-like object.")
    return state


def infer_state_dtype(state_dict: dict[str, torch.Tensor]) -> torch.dtype:
    for value in state_dict.values():
        if isinstance(value, torch.Tensor) and value.is_floating_point():
            return value.dtype
    return torch.float64


def find_experiment_files(results_dir: Path) -> list[ExperimentFile]:
    experiment_files: list[ExperimentFile] = []
    for report_path in sorted(results_dir.glob("*/optimization_report.json")):
        experiment_dir = report_path.parent
        model_path = experiment_dir / "mlp_optimizer_state_dict.pt"
        if not model_path.exists():
            print(f"[skip] missing model state_dict: {model_path}")
            continue
        report = load_json(report_path)
        experiment_files.append(ExperimentFile(experiment_dir, report_path, model_path, report))
    if not experiment_files:
        raise FileNotFoundError(f"No experiment subdirectories found under {results_dir}.")
    return experiment_files


def extract_unique_grid_specs(experiment_files: Sequence[ExperimentFile]) -> list[RegularGridSpec]:
    specs: dict[tuple[int, int], RegularGridSpec] = {}
    for exp in experiment_files:
        cfg = exp.report.get("config", {})
        points_per_axis = int(cfg.get("points_per_axis", -1))
        actual_dataset_size = int(cfg.get("actual_dataset_size", -1))
        target_dataset_size = int(cfg.get("target_dataset_size", actual_dataset_size))
        radius = float(cfg.get("sampling_radius_per_axis", cfg.get("sampling_radius", 0.01)))
        axis_spacing = float(cfg.get("axis_spacing", (2.0 * radius) / max(points_per_axis - 1, 1)))
        if points_per_axis <= 0 or actual_dataset_size <= 0:
            continue
        specs[(points_per_axis, actual_dataset_size)] = RegularGridSpec(
            target_dataset_size, actual_dataset_size, points_per_axis, radius, axis_spacing
        )
    return sorted(specs.values(), key=lambda s: s.actual_dataset_size)


def parse_dataset_size_from_name(name: str) -> int | None:
    match = re.search(r"num_samples_(\d+)", name)
    return int(match.group(1)) if match else None


experiment_files = find_experiment_files(RESULTS_DIR)
grid_specs = extract_unique_grid_specs(experiment_files)
print("num experiments:", len(experiment_files))
print("unique training grids:")
for spec in grid_specs:
    print(spec)

num experiments: 42
unique training grids:
RegularGridSpec(target_dataset_size=8, actual_dataset_size=8, points_per_axis=2, sampling_radius=0.01, axis_spacing=0.02)
RegularGridSpec(target_dataset_size=64, actual_dataset_size=64, points_per_axis=4, sampling_radius=0.01, axis_spacing=0.006666666666666667)
RegularGridSpec(target_dataset_size=216, actual_dataset_size=216, points_per_axis=6, sampling_radius=0.01, axis_spacing=0.004)
RegularGridSpec(target_dataset_size=1000, actual_dataset_size=1000, points_per_axis=10, sampling_radius=0.01, axis_spacing=0.0022222222222222222)
RegularGridSpec(target_dataset_size=10648, actual_dataset_size=10648, points_per_axis=22, sampling_radius=0.01, axis_spacing=0.0009523809523809524)
RegularGridSpec(target_dataset_size=97336, actual_dataset_size=97336, points_per_axis=46, sampling_radius=0.01, axis_spacing=0.00044444444444444447)
RegularGridSpec(target_dataset_size=1000000, actual_dataset_size=1000000, points_per_axis=100, sampling_radius=0.01, axis_spa

## 4. 恢复全局物理问题，并检查 `p_n` 在测试区域中的位置

如果 `p_n` 本身就在 `y*` 附近的测试 cube 内，那么 `p_n` 的单点表现与测试集平均表现接近是可能的。


In [9]:
first_cfg = experiment_files[0].report.get("config", {})
first_state = safe_torch_load_state_dict(experiment_files[0].model_path)
global_dtype = infer_state_dtype(first_state)

p_n_cpu = torch.tensor(first_cfg.get("p_n", [3.0, 4.0, 5.0]), dtype=global_dtype)
v_n_cpu = torch.tensor(first_cfg.get("v_n", [0.5, -0.5, 0.0]), dtype=global_dtype)
m = float(first_cfg.get("m", 1.0))
g = float(first_cfg.get("g", 9.8))
dt = float(first_cfg.get("dt", 0.01))
y_star_cpu = torch.tensor(
    first_cfg.get("y_star", (p_n_cpu + dt * v_n_cpu - dt**2 * torch.tensor([0.0, 0.0, g], dtype=global_dtype)).tolist()),
    dtype=global_dtype,
)
sampling_radius = float(first_cfg.get("sampling_radius_per_axis", first_cfg.get("sampling_radius", 0.01)))

pn_offset = p_n_cpu - y_star_cpu
print("dtype:", global_dtype)
print("p_n:", p_n_cpu.tolist())
print("v_n:", v_n_cpu.tolist())
print("m,g,dt:", m, g, dt)
print("y_star:", y_star_cpu.tolist())
print("sampling_radius:", sampling_radius)
print("p_n - y_star:", pn_offset.tolist())
print("||p_n-y_star||:", float(torch.linalg.vector_norm(pn_offset)))
print("p_n is inside y_star-centered cube:", bool(torch.all(torch.abs(pn_offset) <= sampling_radius)))

dtype: torch.float64
p_n: [3.0, 4.0, 5.0]
v_n: [0.5, -0.5, 0.0]
m,g,dt: 1.0 9.8 0.01
y_star: [3.005, 3.995, 4.99902]
sampling_radius: 0.01
p_n - y_star: [-0.004999999999999893, 0.004999999999999893, 0.000980000000000203]
||p_n-y_star||: 0.0071386553355655335
p_n is inside y_star-centered cube: True


/data/zhoucy/anaconda3/envs/hood/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


## 5. 重建训练集 set distribution：只画分布，不评估训练集误差

训练集是以 `y*` 为中心的规则网格：

$$
S_{\mathrm{train}}
=
\left\{\, y^*+(a_i,a_j,a_k) \;\middle|\; a_i,a_j,a_k \in \mathrm{linspace}(-R,R,n) \,\right\}.
$$

这里我们只抽样一部分训练点用于可视化，不对训练集计算 residual/loss。


In [10]:
def sample_regular_grid_points_for_plot(*, grid_spec: RegularGridSpec, y_star: np.ndarray, max_points: int) -> np.ndarray:
    n = int(grid_spec.points_per_axis)
    total = int(grid_spec.actual_dataset_size)
    radius = float(grid_spec.sampling_radius)
    spacing = float(grid_spec.axis_spacing)
    lower = y_star - radius
    if total <= max_points:
        flat_indices = np.arange(total, dtype=np.int64)
    else:
        flat_indices = np.linspace(0, total - 1, max_points).round().astype(np.int64)
    n2 = n * n
    ix = flat_indices // n2
    rem = flat_indices % n2
    iy = rem // n
    iz = rem % n
    points = np.empty((flat_indices.shape[0], 3), dtype=float)
    points[:, 0] = lower[0] + ix * spacing
    points[:, 1] = lower[1] + iy * spacing
    points[:, 2] = lower[2] + iz * spacing
    return points


def finite_rows(points: np.ndarray) -> np.ndarray:
    points = np.asarray(points, dtype=float).reshape(-1, 3)
    return points[np.isfinite(points).all(axis=1)]


def set_equal_3d_axes(ax, points: np.ndarray) -> None:
    points = finite_rows(points)
    if points.shape[0] == 0:
        points = np.zeros((1, 3), dtype=float)
    center = points.mean(axis=0)
    radius = max(float(np.ptp(points, axis=0).max()) / 2.0, 1e-8)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)

## 6. 构造 held-out 测试集，并显式剔除训练集规则网格点

测试集从 `y*` 附近 cube 随机采样，然后检查每个点是否落在任意训练规则网格上。若重合则剔除。

注意：连续随机采样与规则网格精确重合的概率本来就接近 0，但这里仍然做显式剔除，保证测试集不含训练样本。


In [11]:
def points_in_grid_mask(points: torch.Tensor, *, grid_spec: RegularGridSpec, y_star: torch.Tensor, tol: float = GRID_MATCH_TOL) -> torch.Tensor:
    n = int(grid_spec.points_per_axis)
    radius = float(grid_spec.sampling_radius)
    spacing = float(grid_spec.axis_spacing)
    lower = y_star.to(dtype=points.dtype).unsqueeze(0) - radius
    coords = (points - lower) / spacing
    rounded = torch.round(coords)
    close_to_integer = torch.abs(coords - rounded) <= tol
    in_range = (rounded >= 0) & (rounded <= (n - 1))
    return torch.all(close_to_integer & in_range, dim=1)


def build_fixed_heldout_test_set_excluding_training_grids(*, y_star: torch.Tensor, radius: float, num_random: int, seed: int,
                                                         radius_scale: float, dtype: torch.dtype, grid_specs: Sequence[RegularGridSpec]):
    effective_radius = float(radius) * float(radius_scale)
    generator = torch.Generator(device="cpu")
    generator.manual_seed(int(seed))
    y_star_cpu = y_star.detach().cpu().to(dtype=dtype)

    collected_chunks = []
    num_collected = 0
    num_candidates_generated = 0
    num_rejected_overlap = 0
    batch_candidates = max(2048, min(65536, num_random * 2))

    while num_collected < num_random:
        remaining = num_random - num_collected
        current_batch = max(batch_candidates, remaining * 2)
        offsets = (2.0 * torch.rand((current_batch, 3), generator=generator, dtype=dtype) - 1.0) * effective_radius
        candidates = y_star_cpu.unsqueeze(0) + offsets
        keep_mask = torch.ones(current_batch, dtype=torch.bool)
        for grid_spec in grid_specs:
            keep_mask &= ~points_in_grid_mask(candidates, grid_spec=grid_spec, y_star=y_star_cpu)
        kept = candidates[keep_mask]
        num_candidates_generated += int(current_batch)
        num_rejected_overlap += int((~keep_mask).sum().item())
        if kept.shape[0] > remaining:
            kept = kept[:remaining]
        if kept.numel() > 0:
            collected_chunks.append(kept)
            num_collected += int(kept.shape[0])

    test_points = torch.cat(collected_chunks, dim=0)
    metadata = {
        "mode": "uniform_random_cube_near_y_star_excluding_all_training_grids",
        "num_random_points": int(num_random),
        "num_total_points": int(test_points.shape[0]),
        "seed": int(seed),
        "sampling_center": "y_star",
        "base_sampling_radius": float(radius),
        "radius_scale": float(radius_scale),
        "effective_sampling_radius": effective_radius,
        "strictly_excludes_all_training_points": True,
        "num_candidate_points_generated": int(num_candidates_generated),
        "num_candidate_points_rejected_due_to_training_overlap": int(num_rejected_overlap),
        "num_unique_training_grids_checked": int(len(grid_specs)),
    }
    return test_points, metadata


test_points_cpu, test_metadata = build_fixed_heldout_test_set_excluding_training_grids(
    y_star=y_star_cpu,
    radius=sampling_radius,
    num_random=NUM_TEST,
    seed=TEST_SEED,
    radius_scale=RADIUS_SCALE,
    dtype=global_dtype,
    grid_specs=grid_specs,
)
print(json.dumps(test_metadata, indent=2, ensure_ascii=False))

# 严格检查是否与任何训练网格重合
overlap_counts = []
for spec in grid_specs:
    overlap_counts.append((spec.actual_dataset_size, int(points_in_grid_mask(test_points_cpu, grid_spec=spec, y_star=y_star_cpu).sum().item())))
overlap_counts

{
  "mode": "uniform_random_cube_near_y_star_excluding_all_training_grids",
  "num_random_points": 4096,
  "num_total_points": 4096,
  "seed": 20260617,
  "sampling_center": "y_star",
  "base_sampling_radius": 0.01,
  "radius_scale": 1.0,
  "effective_sampling_radius": 0.01,
  "strictly_excludes_all_training_points": true,
  "num_candidate_points_generated": 8192,
  "num_candidate_points_rejected_due_to_training_overlap": 0,
  "num_unique_training_grids_checked": 7
}


[(8, 0), (64, 0), (216, 0), (1000, 0), (10648, 0), (97336, 0), (1000000, 0)]

## 7. 画训练集和测试集的 set distribution

这个图只展示训练集、测试集、`p_n`、`y*` 的空间分布，不评估训练集 residual/loss。


In [12]:
def plot_training_and_test_distribution_overview(save_path: Path) -> None:
    y_star_np = np.asarray(y_star_cpu.tolist(), dtype=float)
    p_n_np = np.asarray(p_n_cpu.tolist(), dtype=float)
    test_points = test_points_cpu.detach().cpu().numpy()
    if test_points.shape[0] > MAX_TEST_POINTS_FOR_DISTRIBUTION:
        idx = np.linspace(0, test_points.shape[0] - 1, MAX_TEST_POINTS_FOR_DISTRIBUTION).round().astype(int)
        test_points = test_points[idx]

    num_plots = len(grid_specs) + 1
    num_cols = min(4, num_plots)
    num_rows = math.ceil(num_plots / num_cols)
    fig = plt.figure(figsize=(5.2 * num_cols, 4.9 * num_rows))

    for i, spec in enumerate(grid_specs):
        ax = fig.add_subplot(num_rows, num_cols, i + 1, projection="3d")
        train_points = sample_regular_grid_points_for_plot(grid_spec=spec, y_star=y_star_np, max_points=MAX_TRAINING_POINTS_PER_SUBPLOT)
        ax.scatter(train_points[:,0], train_points[:,1], train_points[:,2], s=5, alpha=0.35, color="C0", label=f"training ({train_points.shape[0]}/{spec.actual_dataset_size})")
        ax.scatter(*p_n_np, marker="x", s=120, linewidths=2.0, color="C3", label=r"$p_n$")
        ax.scatter(*y_star_np, marker="*", s=220, color="C2", label=r"$y^*$")
        set_equal_3d_axes(ax, np.vstack([train_points, p_n_np[None, :], y_star_np[None, :]]))
        ax.set_title(f"Training set\nN={spec.actual_dataset_size:,}, axis={spec.points_per_axis}")
        ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
        ax.legend(fontsize=7)

    ax = fig.add_subplot(num_rows, num_cols, len(grid_specs) + 1, projection="3d")
    ax.scatter(test_points[:,0], test_points[:,1], test_points[:,2], s=5, alpha=0.35, color="C1", label=f"held-out test ({test_points.shape[0]}/{test_points_cpu.shape[0]})")
    ax.scatter(*p_n_np, marker="x", s=120, linewidths=2.0, color="C3", label=r"$p_n$")
    ax.scatter(*y_star_np, marker="*", s=220, color="C2", label=r"$y^*$")
    set_equal_3d_axes(ax, np.vstack([test_points, p_n_np[None, :], y_star_np[None, :]]))
    ax.set_title("Held-out test set\n(excludes training-grid points)")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.legend(fontsize=7)

    fig.suptitle("Set distribution overview: training grids and held-out test set\ntraining=C0, test=C1, p_n=C3, y*=C2", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_training_and_test_distribution_overview(OUTPUT_DIR / "training_and_test_set_distribution_overview.png")

## 8. 加载模型参数

这一步会恢复网络结构、归一化参数和训练好的权重。注意 `input_mean/input_std` 和模型权重必须来自同一实验目录。


In [13]:
def instantiate_model_from_report_and_state(*, experiment: ExperimentFile, device: torch.device):
    state_dict = safe_torch_load_state_dict(experiment.model_path)
    dtype = infer_state_dtype(state_dict)
    cfg = experiment.report.get("config", {})
    input_mean = torch.tensor(cfg.get("input_mean", [0.0] * 12), dtype=dtype)
    input_std = torch.tensor(cfg.get("input_std", [1.0] * 12), dtype=dtype)
    model = MLPOptimizer(
        dtype=dtype,
        use_input_normalization=cfg.get("use_input_normalization", True),
        use_output_dt_scaling=cfg.get("use_output_dt_scaling", True),
        input_mean=input_mean,
        input_std=input_std,
    ).to(device=device, dtype=dtype)
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    p_n = torch.tensor(cfg.get("p_n", [3.0, 4.0, 5.0]), dtype=dtype)
    v_n = torch.tensor(cfg.get("v_n", [0.5, -0.5, 0.0]), dtype=dtype)
    return model, dtype, {"p_n": p_n, "v_n": v_n}

sample_model, sample_dtype, sample_tensors = instantiate_model_from_report_and_state(experiment=experiment_files[0], device=device)
print("sample:", experiment_files[0].experiment_dir.name)
print("dtype:", sample_dtype)
print("input_mean:", sample_model.input_mean.detach().cpu().numpy())
print("input_std :", sample_model.input_std.detach().cpu().numpy())

sample: adam_lr_1e-02_grid_axis_100_num_samples_1000000
dtype: torch.float64
input_mean: [ 3.005    3.995    4.99902  3.       4.       5.       0.5     -0.5
  0.       1.       9.8      0.01   ]
input_std : [0.00583153 0.00583153 0.00583153 1.         1.         1.
 1.         1.         1.         1.         1.         1.        ]


## 9. 测试函数：对初始点集合计算 residual 和 loss gap

对每个测试初始点 $y_0$，网络迭代展开为：

$$
y^{(k+1)}=y^{(k)}+\Delta y_\theta\left(y^{(k)},p_n,v_n,m,g,dt\right).
$$

每一步记录 residual：

$$
r_k(y_0)=\left\lVert \nabla E\left(y^{(k)}(y_0)\right) \right\rVert_2,
$$

以及 energy loss gap：

$$
\ell_k(y_0)=E\left(y^{(k)}(y_0)\right)-E(y^*).
$$

对测试集计算 mean / median / p95 / max；对 `p_n` 则只输入一个点，得到单点指标。


In [14]:
@torch.no_grad()
def evaluate_model_on_initial_set(*, model: MLPOptimizer, initial_points_cpu: torch.Tensor, p_n: torch.Tensor, v_n: torch.Tensor,
                                  y_star: torch.Tensor, m: float, g: float, dt: float, steps: int,
                                  batch_size: int, device: torch.device, dtype: torch.dtype):
    model.eval()
    p_n_device = p_n.to(device=device, dtype=dtype)
    v_n_device = v_n.to(device=device, dtype=dtype)
    y_star_device = y_star.to(device=device, dtype=dtype)
    history = torch.cat([p_n_device, v_n_device], dim=0)
    params = torch.tensor([m, g, dt], device=device, dtype=dtype)
    e_star = variational_energy(y_star_device, p_n_device, v_n_device, m, g, dt)

    all_residuals = []
    all_gaps = []
    num_points = int(initial_points_cpu.shape[0])
    for start in range(0, num_points, batch_size):
        end = min(start + batch_size, num_points)
        y = initial_points_cpu[start:end].to(device=device, dtype=dtype)
        batch_residuals = []
        batch_gaps = []
        for step in range(steps + 1):
            residual = stationarity_residual_norm(y, p_n_device, v_n_device, m, g, dt)
            gap = variational_energy(y, p_n_device, v_n_device, m, g, dt) - e_star
            batch_residuals.append(residual.detach().cpu())
            batch_gaps.append(gap.detach().cpu())
            if step == steps:
                break
            y = y + model(y, history, params)
        all_residuals.append(torch.stack(batch_residuals, dim=1))
        all_gaps.append(torch.stack(batch_gaps, dim=1))

    residuals = torch.cat(all_residuals, dim=0).numpy().astype(float)
    gaps = torch.cat(all_gaps, dim=0).numpy().astype(float)
    residuals[~np.isfinite(residuals)] = np.nan
    gaps[~np.isfinite(gaps)] = np.nan

    def stats(prefix: str, values: np.ndarray):
        final_values = values[:, -1]
        return {
            f"{prefix}_mean_by_step": np.nanmean(values, axis=0).tolist(),
            f"{prefix}_median_by_step": np.nanmedian(values, axis=0).tolist(),
            f"{prefix}_p95_by_step": np.nanpercentile(values, 95, axis=0).tolist(),
            f"{prefix}_max_by_step": np.nanmax(values, axis=0).tolist(),
            f"final_{prefix}_mean": float(np.nanmean(final_values)),
            f"final_{prefix}_median": float(np.nanmedian(final_values)),
            f"final_{prefix}_p95": float(np.nanpercentile(final_values, 95)),
            f"final_{prefix}_max": float(np.nanmax(final_values)),
        }

    summary = {"steps": int(steps), "num_points": int(num_points)}
    summary.update(stats("residual", residuals))
    summary.update(stats("loss_gap", gaps))
    if num_points == 1:
        summary["single_point_residual_by_step"] = residuals[0].tolist()
        summary["single_point_loss_gap_by_step"] = gaps[0].tolist()
        summary["single_point_final_residual"] = float(residuals[0, -1])
        summary["single_point_final_loss_gap"] = float(gaps[0, -1])
    return summary, residuals, gaps

## 10. 诊断 1：不跑网络，先看测试集初始分布和 `p_n`

如果 `p_n` 的初始 residual / loss gap 位于测试集分布中间，那么最终也可能和测试集平均值接近。


In [15]:
def diagnostic_initial_test_vs_pn():
    dtype = test_points_cpu.dtype
    p_n_ = p_n_cpu.to(dtype=dtype)
    v_n_ = v_n_cpu.to(dtype=dtype)
    y_star_ = y_star_cpu.to(dtype=dtype)
    e_star_ = variational_energy(y_star_, p_n_, v_n_, m, g, dt)

    test_dist = torch.linalg.vector_norm(test_points_cpu - y_star_.reshape(1, 3), dim=1).numpy()
    pn_dist = float(torch.linalg.vector_norm(p_n_ - y_star_))
    test_res0 = stationarity_residual_norm(test_points_cpu, p_n_, v_n_, m, g, dt).numpy()
    pn_res0 = float(stationarity_residual_norm(p_n_.reshape(1, 3), p_n_, v_n_, m, g, dt)[0])
    test_gap0 = (variational_energy(test_points_cpu, p_n_, v_n_, m, g, dt) - e_star_).numpy()
    pn_gap0 = float(variational_energy(p_n_, p_n_, v_n_, m, g, dt) - e_star_)

    rows = []
    for name, arr, pn_value in [
        ("distance_to_y_star", test_dist, pn_dist),
        ("initial_residual", test_res0, pn_res0),
        ("initial_loss_gap", test_gap0, pn_gap0),
    ]:
        rows.append({
            "metric": name,
            "test_mean": float(np.mean(arr)),
            "test_median": float(np.median(arr)),
            "test_p05": float(np.percentile(arr, 5)),
            "test_p95": float(np.percentile(arr, 95)),
            "pn_value": pn_value,
            "pn_percentile_in_test": float(100.0 * np.mean(arr <= pn_value)),
        })
    return pd.DataFrame(rows), (test_dist, pn_dist, test_res0, pn_res0, test_gap0, pn_gap0)

initial_diag_df, initial_arrays = diagnostic_initial_test_vs_pn()
initial_diag_df

,metric,test_mean,test_median,test_p05,test_p95,pn_value,pn_percentile_in_test
0,distance_to_y_star,0.009570,0.009835,0.004457,0.013820,0.007139,19.482422
1,initial_residual,95.697252,98.350396,44.571349,138.198638,71.386553,19.482422
2,initial_loss_gap,0.496926,0.483640,0.099330,0.954943,0.254802,19.482422


In [16]:
test_dist, pn_dist, test_res0, pn_res0, test_gap0, pn_gap0 = initial_arrays
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, arr, pn_value, title, xlabel in [
    (axes[0], test_dist, pn_dist, "Initial distance to y*", r"$\|y_0-y^*\|_2$"),
    (axes[1], test_res0, pn_res0, "Initial residual", r"$\|\nabla E(y_0)\|_2$"),
    (axes[2], test_gap0, pn_gap0, "Initial loss gap", r"$E(y_0)-E(y^*)$"),
]:
    ax.hist(arr, bins=50, alpha=0.75)
    ax.axvline(pn_value, linestyle="--", linewidth=2.0, label="p_n")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("count")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "diagnostic_initial_test_distribution_vs_pn.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. 批量评估所有模型

这里测试集和 `p_n` 分开输入：

- `test_points_cpu`：全测试集；
- `p_n_cpu.reshape(1,3)`：单点 `p_n`。

如果它们结果完全一样，这里能通过 raw cache 继续检查每个测试点的最终分布。


In [17]:
def optimizer_key(record: dict[str, Any]) -> tuple[str, float]:
    return str(record["optimizer_name"]).lower(), float(record["learning_rate"])


def optimizer_label(record_or_key: dict[str, Any] | tuple[str, float]) -> str:
    if isinstance(record_or_key, tuple):
        name, lr = record_or_key
    else:
        name, lr = optimizer_key(record_or_key)
    return f"{name.upper()} lr={lr:.0e}"


def unique_optimizer_keys(records: Sequence[dict[str, Any]]) -> list[tuple[str, float]]:
    keys = sorted({optimizer_key(r) for r in records}, key=lambda item: (item[0], item[1]))
    preferred_order = {"sgd": 0, "adam": 1}
    return sorted(keys, key=lambda item: (preferred_order.get(item[0], 99), -item[1]))

records = []
raw_eval_cache = {}
for exp in experiment_files:
    cfg = exp.report.get("config", {})
    print("[eval]", exp.experiment_dir.name)
    model, dtype, tensors = instantiate_model_from_report_and_state(experiment=exp, device=device)
    p_n = tensors["p_n"]
    v_n = tensors["v_n"]
    m_i = float(cfg.get("m", m))
    g_i = float(cfg.get("g", g))
    dt_i = float(cfg.get("dt", dt))

    if not torch.allclose(p_n.to(global_dtype), p_n_cpu.to(global_dtype)) or not torch.allclose(v_n.to(global_dtype), v_n_cpu.to(global_dtype)):
        raise ValueError(f"{exp.experiment_dir.name} uses different p_n/v_n.")
    if (m_i, g_i, dt_i) != (m, g, dt):
        raise ValueError(f"{exp.experiment_dir.name} uses different m/g/dt.")

    test_summary, test_residuals, test_gaps = evaluate_model_on_initial_set(
        model=model,
        initial_points_cpu=test_points_cpu.to(dtype=dtype),
        p_n=p_n,
        v_n=v_n,
        y_star=y_star_cpu.to(dtype=dtype),
        m=m_i,
        g=g_i,
        dt=dt_i,
        steps=STEPS,
        batch_size=BATCH_SIZE,
        device=device,
        dtype=dtype,
    )
    pn_summary, pn_residuals, pn_gaps = evaluate_model_on_initial_set(
        model=model,
        initial_points_cpu=p_n_cpu.reshape(1, 3).to(dtype=dtype),
        p_n=p_n,
        v_n=v_n,
        y_star=y_star_cpu.to(dtype=dtype),
        m=m_i,
        g=g_i,
        dt=dt_i,
        steps=STEPS,
        batch_size=1,
        device=device,
        dtype=dtype,
    )

    dataset_size = int(cfg.get("actual_dataset_size", parse_dataset_size_from_name(exp.experiment_dir.name) or -1))
    record = {
        "experiment_name": cfg.get("experiment_name", exp.experiment_dir.name),
        "experiment_dir": str(exp.experiment_dir),
        "optimizer_name": cfg.get("optimizer_name", "unknown"),
        "learning_rate": float(cfg.get("learning_rate", float("nan"))),
        "target_dataset_size": int(cfg.get("target_dataset_size", dataset_size)),
        "dataset_size": dataset_size,
        "points_per_axis": int(cfg.get("points_per_axis", -1)),
        "test_num_points": int(test_summary["num_points"]),
        "residual_mean_by_step": test_summary["residual_mean_by_step"],
        "residual_median_by_step": test_summary["residual_median_by_step"],
        "residual_p95_by_step": test_summary["residual_p95_by_step"],
        "loss_gap_mean_by_step": test_summary["loss_gap_mean_by_step"],
        "loss_gap_median_by_step": test_summary["loss_gap_median_by_step"],
        "loss_gap_p95_by_step": test_summary["loss_gap_p95_by_step"],
        "final_residual_mean": test_summary["final_residual_mean"],
        "final_residual_median": test_summary["final_residual_median"],
        "final_residual_p95": test_summary["final_residual_p95"],
        "final_residual_max": test_summary["final_residual_max"],
        "final_loss_gap_mean": test_summary["final_loss_gap_mean"],
        "final_loss_gap_median": test_summary["final_loss_gap_median"],
        "final_loss_gap_p95": test_summary["final_loss_gap_p95"],
        "final_loss_gap_max": test_summary["final_loss_gap_max"],
        "pn_residual_by_step": pn_summary["single_point_residual_by_step"],
        "pn_loss_gap_by_step": pn_summary["single_point_loss_gap_by_step"],
        "pn_final_residual": pn_summary["single_point_final_residual"],
        "pn_final_loss_gap": pn_summary["single_point_final_loss_gap"],
    }
    records.append(record)
    raw_eval_cache[record["experiment_name"]] = {
        "test_residuals": test_residuals,
        "test_gaps": test_gaps,
        "pn_residuals": pn_residuals,
        "pn_gaps": pn_gaps,
        "experiment": exp,
    }

records = sorted(records, key=lambda r: (optimizer_key(r), int(r["dataset_size"])))
df = pd.DataFrame(records)
df[["optimizer_name", "learning_rate", "dataset_size", "final_residual_mean", "pn_final_residual", "final_loss_gap_mean", "pn_final_loss_gap"]]

[eval] adam_lr_1e-02_grid_axis_100_num_samples_1000000
[eval] adam_lr_1e-02_grid_axis_10_num_samples_1000
[eval] adam_lr_1e-02_grid_axis_22_num_samples_10648
[eval] adam_lr_1e-02_grid_axis_2_num_samples_8
[eval] adam_lr_1e-02_grid_axis_46_num_samples_97336
[eval] adam_lr_1e-02_grid_axis_4_num_samples_64
[eval] adam_lr_1e-02_grid_axis_6_num_samples_216
[eval] adam_lr_1e-03_grid_axis_100_num_samples_1000000
[eval] adam_lr_1e-03_grid_axis_10_num_samples_1000
[eval] adam_lr_1e-03_grid_axis_22_num_samples_10648
[eval] adam_lr_1e-03_grid_axis_2_num_samples_8
[eval] adam_lr_1e-03_grid_axis_46_num_samples_97336
[eval] adam_lr_1e-03_grid_axis_4_num_samples_64
[eval] adam_lr_1e-03_grid_axis_6_num_samples_216
[eval] adam_lr_1e-04_grid_axis_100_num_samples_1000000
[eval] adam_lr_1e-04_grid_axis_10_num_samples_1000
[eval] adam_lr_1e-04_grid_axis_22_num_samples_10648
[eval] adam_lr_1e-04_grid_axis_2_num_samples_8
[eval] adam_lr_1e-04_grid_axis_46_num_samples_97336
[eval] adam_lr_1e-04_grid_axis_4_nu

,optimizer_name,learning_rate,dataset_size,final_residual_mean,pn_final_residual,final_loss_gap_mean,pn_final_loss_gap
0,adam,0.0001,8,1.786616e-03,1.786616e-03,1.595950e-10,1.595950e-10
1,adam,0.0001,64,3.818218e-03,3.818218e-03,7.289316e-10,7.289316e-10
2,adam,0.0001,216,6.569473e-04,6.569473e-04,2.157208e-11,2.157208e-11
3,adam,0.0001,1000,2.886508e-03,2.886508e-03,4.165912e-10,4.165912e-10
4,adam,0.0001,10648,7.436092e-03,7.436092e-03,2.764764e-09,2.764764e-09
5,adam,0.0001,97336,1.091929e-02,1.091929e-02,5.961532e-09,5.961532e-09
6,adam,0.0001,1000000,6.469770e-03,6.469770e-03,2.092889e-09,2.092889e-09
7,adam,0.0010,8,5.386199e-02,5.386199e-02,1.450557e-07,1.450557e-07
8,adam,0.0010,64,1.835096e-04,1.835096e-04,1.676881e-12,1.676881e-12
9,adam,0.0010,216,8.085786e-04,8.085786e-04,3.267786e-11,3.267786e-11


## 12. 汇总图：测试集和 `p_n` 的 residual/loss

第一行 residual，第二行 energy loss gap。前三列是测试集统计量，最后一列是 `p_n` 单点。


In [18]:
def plot_final_residual_loss_vs_dataset_size(records: Sequence[dict[str, Any]], save_path: Path) -> None:
    fig, axes = plt.subplots(2, 4, figsize=(22, 10.8))
    residual_metrics = [
        ("final_residual_mean", "Test mean residual", r"$\frac{1}{|T|}\sum r_K(y_0)$"),
        ("final_residual_median", "Test median residual", r"$\mathrm{median}\, r_K(y_0)$"),
        ("final_residual_p95", "Test p95 residual", r"$\mathrm{p95}\, r_K(y_0)$"),
        ("pn_final_residual", r"$p_n$ residual", r"$r_K(p_n)$"),
    ]
    loss_metrics = [
        ("final_loss_gap_mean", "Test mean loss gap", r"$\frac{1}{|T|}\sum \ell_K(y_0)$"),
        ("final_loss_gap_median", "Test median loss gap", r"$\mathrm{median}\, \ell_K(y_0)$"),
        ("final_loss_gap_p95", "Test p95 loss gap", r"$\mathrm{p95}\, \ell_K(y_0)$"),
        ("pn_final_loss_gap", r"$p_n$ loss gap", r"$\ell_K(p_n)$"),
    ]
    for key in unique_optimizer_keys(records):
        selected = sorted([r for r in records if optimizer_key(r) == key], key=lambda r: int(r["dataset_size"]))
        sizes = [int(r["dataset_size"]) for r in selected]
        for ax, (metric_name, title, ylabel) in zip(axes[0], residual_metrics):
            ax.plot(sizes, [finite_plot_value(r.get(metric_name)) for r in selected], marker="o", label=optimizer_label(key))
            ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(title); ax.set_xlabel("training dataset size"); ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3)
        for ax, (metric_name, title, ylabel) in zip(axes[1], loss_metrics):
            ax.plot(sizes, [finite_plot_value(r.get(metric_name)) for r in selected], marker="o", label=optimizer_label(key))
            ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(title); ax.set_xlabel("training dataset size"); ax.set_ylabel(ylabel); ax.grid(True, alpha=0.3)
    for row in axes:
        for ax in row:
            ax.legend(fontsize=7)
    fig.suptitle("Held-out test set and p_n: residual / energy loss gap", y=1.01)
    fig.text(0.5, -0.01, r"$r_k=\|\nabla E(y^{(k)})\|_2$, $\ell_k=E(y^{(k)})-E(y^*)$. First three columns: test set; last column: p_n.", ha="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_final_residual_loss_vs_dataset_size(records, OUTPUT_DIR / "heldout_final_residual_loss_vs_training_dataset_size.png")

## 13. 诊断 2：直接打印 `test mean` 与 `p_n` 的相对差

如果 log 图看不出区别，不代表数值完全一样。这里直接算相对差：

$$
\frac{\left|m_{p_n}-m_{\mathrm{test\ mean}}\right|}{\left|m_{\mathrm{test\ mean}}\right|+\varepsilon}.
$$

其中 $m$ 可以是 final residual，也可以是 final loss gap。分母中加入一个很小的 $\varepsilon$ 是为了避免除以 0。


In [19]:
diag_rows = []
for r in records:
    residual_mean = float(r["final_residual_mean"])
    pn_residual = float(r["pn_final_residual"])
    loss_mean = float(r["final_loss_gap_mean"])
    pn_loss = float(r["pn_final_loss_gap"])
    diag_rows.append({
        "optimizer": r["optimizer_name"],
        "lr": r["learning_rate"],
        "N": r["dataset_size"],
        "test_residual_mean": residual_mean,
        "pn_residual": pn_residual,
        "residual_rel_diff": abs(pn_residual - residual_mean) / max(abs(residual_mean), 1e-300),
        "test_loss_mean": loss_mean,
        "pn_loss": pn_loss,
        "loss_rel_diff": abs(pn_loss - loss_mean) / max(abs(loss_mean), 1e-300),
    })

diag_df = pd.DataFrame(diag_rows)
diag_df.sort_values(["optimizer", "lr", "N"])

,optimizer,lr,N,test_residual_mean,pn_residual,residual_rel_diff,test_loss_mean,pn_loss,loss_rel_diff
0,adam,0.0001,8,1.786616e-03,1.786616e-03,1.656847e-09,1.595950e-10,1.595950e-10,0.000000e+00
1,adam,0.0001,64,3.818218e-03,3.818218e-03,2.271640e-16,7.289316e-10,7.289316e-10,0.000000e+00
2,adam,0.0001,216,6.569473e-04,6.569473e-04,9.706752e-10,2.157208e-11,2.157208e-11,0.000000e+00
3,adam,0.0001,1000,2.886508e-03,2.886508e-03,3.004883e-16,4.165912e-10,4.165912e-10,0.000000e+00
4,adam,0.0001,10648,7.436092e-03,7.436092e-03,2.332843e-16,2.764764e-09,2.764764e-09,0.000000e+00
5,adam,0.0001,97336,1.091929e-02,1.091929e-02,1.588678e-16,5.961532e-09,5.961532e-09,0.000000e+00
6,adam,0.0001,1000000,6.469770e-03,6.469770e-03,1.340638e-16,2.092889e-09,2.092889e-09,0.000000e+00
7,adam,0.0010,8,5.386199e-02,5.386199e-02,2.364438e-08,1.450557e-07,1.450557e-07,1.133715e-08
8,adam,0.0010,64,1.835096e-04,1.835096e-04,8.316886e-09,1.676881e-12,1.676881e-12,0.000000e+00
9,adam,0.0010,216,8.085786e-04,8.085786e-04,1.340874e-16,3.267786e-11,3.267786e-11,0.000000e+00


In [20]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for key in unique_optimizer_keys(records):
    selected = diag_df[(diag_df["optimizer"].str.lower() == key[0]) & (np.isclose(diag_df["lr"], key[1]))].sort_values("N")
    axes[0].plot(selected["N"], selected["residual_rel_diff"], marker="o", label=optimizer_label(key))
    axes[1].plot(selected["N"], selected["loss_rel_diff"], marker="o", label=optimizer_label(key))
for ax, title in zip(axes, ["Relative difference: residual", "Relative difference: loss gap"]):
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("training dataset size")
    ax.set_ylabel(r"$|p_n-\mathrm{test\ mean}|/|\mathrm{test\ mean}|$")
    ax.set_title(title); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "diagnostic_pn_vs_test_mean_relative_difference.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. 诊断 3：看最终测试集分布中 `p_n` 的位置

默认选 Adam lr=1e-4、最接近 10000 点训练集的模型。若 `p_n` 的最终误差位于测试集分布中间，那么 `p_n` 和测试集均值接近是合理的。


In [21]:
def find_best_record(records: Sequence[dict[str, Any]], *, optimizer_name: str, learning_rate: float, target_dataset_size: int) -> dict[str, Any]:
    candidates = []
    for r in records:
        if str(r["optimizer_name"]).lower() != optimizer_name.lower():
            continue
        if not math.isclose(float(r["learning_rate"]), learning_rate, rel_tol=0.0, abs_tol=learning_rate * 1e-8):
            continue
        candidates.append((abs(int(r["dataset_size"]) - target_dataset_size), int(r["dataset_size"]), r))
    if not candidates:
        raise ValueError("No matching record.")
    candidates.sort(key=lambda x: (x[0], x[1]))
    return candidates[0][2]

diagnostic_record = find_best_record(records, optimizer_name="adam", learning_rate=1e-4, target_dataset_size=10_000)
cache = raw_eval_cache[diagnostic_record["experiment_name"]]
final_test_res = cache["test_residuals"][:, -1]
final_test_gap = cache["test_gaps"][:, -1]
final_pn_res = float(cache["pn_residuals"][0, -1])
final_pn_gap = float(cache["pn_gaps"][0, -1])
print(diagnostic_record["experiment_name"], diagnostic_record["dataset_size"])
print("p_n residual percentile:", 100.0 * np.mean(final_test_res <= final_pn_res))
print("p_n loss percentile:", 100.0 * np.mean(final_test_gap <= final_pn_gap))

adam_lr_1e-04_grid_axis_22_num_samples_10648 10648
p_n residual percentile: 100.0
p_n loss percentile: 100.0


In [22]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(final_test_res[np.isfinite(final_test_res)], bins=60, alpha=0.75)
axes[0].axvline(final_pn_res, linestyle="--", linewidth=2, label="p_n")
axes[0].set_xscale("log"); axes[0].set_title("Final test residual distribution"); axes[0].set_xlabel(r"$r_K(y_0)$"); axes[0].set_ylabel("count"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].hist(np.maximum(final_test_gap[np.isfinite(final_test_gap)], PLOT_FLOOR), bins=60, alpha=0.75)
axes[1].axvline(max(final_pn_gap, PLOT_FLOOR), linestyle="--", linewidth=2, label="p_n")
axes[1].set_xscale("log"); axes[1].set_title("Final test loss-gap distribution"); axes[1].set_xlabel(r"$\ell_K(y_0)$"); axes[1].set_ylabel("count"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "diagnostic_final_distribution_vs_pn.png", dpi=300, bbox_inches="tight")
plt.show()

## 15. 一致性检查：loss gap 与 residual 是否满足理论关系

如果下面的误差很小，说明 residual 和 loss 计算逻辑基本没问题：

$$
\ell(y)=\frac{dt^2}{2m}\,r(y)^2.
$$

其中：

$$
r(y)=\lVert \nabla E(y) \rVert_2,
\qquad
\ell(y)=E(y)-E(y^*).
$$


In [23]:
def check_residual_loss_identity(residuals: np.ndarray, gaps: np.ndarray, m: float, dt: float) -> pd.DataFrame:
    predicted_gaps = (dt**2 / (2.0 * m)) * residuals**2
    abs_err = np.abs(gaps - predicted_gaps)
    rel_err = abs_err / np.maximum(np.abs(gaps), 1e-300)
    finite = np.isfinite(abs_err) & np.isfinite(rel_err)
    return pd.DataFrame([{
        "max_abs_error": float(np.nanmax(abs_err[finite])),
        "mean_abs_error": float(np.nanmean(abs_err[finite])),
        "max_rel_error": float(np.nanmax(rel_err[finite])),
        "mean_rel_error": float(np.nanmean(rel_err[finite])),
    }])

identity_check_df = check_residual_loss_identity(cache["test_residuals"], cache["test_gaps"], m=m, dt=dt)
identity_check_df

,max_abs_error,mean_abs_error,max_rel_error,mean_rel_error
0,1.310063e-14,8.251662e-15,0.000005,0.000003


## 16. 详细轨迹图：Adam lr=1e-4, N=8 和 N≈10000

下面保留原脚本中的详细图逻辑：从测试集中选 3 个点，对 Adam `lr=1e-4` 的两个训练规模分别画：

- `final_reference_residual_comparison.png`
- `final_reference_energy_contour_2d.png`
- `final_reference_trajectory_3d.png`

为了 notebook 简洁，这里直接给出完整函数。运行后结果保存在 `OUTPUT_DIR / detailed_test_points`。


In [24]:
@torch.no_grad()
def evaluate_single_trajectory(model: MLPOptimizer, initial_y: torch.Tensor, p_n: torch.Tensor, v_n: torch.Tensor, y_star: torch.Tensor,
                               m: float, g: float, dt: float, steps: int, device: torch.device, dtype: torch.dtype) -> dict[str, Any]:
    p_n_device = p_n.to(device=device, dtype=dtype)
    v_n_device = v_n.to(device=device, dtype=dtype)
    y_star_device = y_star.to(device=device, dtype=dtype)
    y = initial_y.to(device=device, dtype=dtype).clone()
    history = torch.cat([p_n_device, v_n_device], dim=0)
    params = torch.tensor([m, g, dt], device=device, dtype=dtype)
    e_star = variational_energy(y_star_device, p_n_device, v_n_device, m, g, dt)
    iterations = []
    for step in range(steps + 1):
        energy = variational_energy(y, p_n_device, v_n_device, m, g, dt)
        residual_norm = stationarity_residual_norm(y, p_n_device, v_n_device, m, g, dt)
        iterations.append({
            "step": int(step),
            "y": [float(x) for x in y.detach().cpu().tolist()],
            "energy": float(energy.detach().cpu().item()),
            "gap": float((energy - e_star).detach().cpu().item()),
            "residual_norm": float(residual_norm.detach().cpu().item()),
        })
        if step == steps:
            break
        delta = model(y, history, params)
        y = y + delta
        iterations[-1]["next_delta_norm"] = float(torch.linalg.vector_norm(delta).detach().cpu().item())
    return {"initial_y": [float(x) for x in initial_y.detach().cpu().tolist()], "iterations": iterations}


@torch.no_grad()
def evaluate_newton_trajectory(initial_y: torch.Tensor, p_n: torch.Tensor, v_n: torch.Tensor, y_star: torch.Tensor,
                               m: float, g: float, dt: float, steps: int, device: torch.device, dtype: torch.dtype) -> dict[str, Any]:
    p_n_device = p_n.to(device=device, dtype=dtype)
    v_n_device = v_n.to(device=device, dtype=dtype)
    y_star_device = y_star.to(device=device, dtype=dtype)
    y = initial_y.to(device=device, dtype=dtype).clone()
    e_star = variational_energy(y_star_device, p_n_device, v_n_device, m, g, dt)
    iterations = []
    for step in range(steps + 1):
        energy = variational_energy(y, p_n_device, v_n_device, m, g, dt)
        residual_norm = stationarity_residual_norm(y, p_n_device, v_n_device, m, g, dt)
        iterations.append({
            "step": int(step),
            "y": [float(x) for x in y.detach().cpu().tolist()],
            "energy": float(energy.detach().cpu().item()),
            "gap": float((energy - e_star).detach().cpu().item()),
            "residual_norm": float(residual_norm.detach().cpu().item()),
        })
        if step == steps:
            break
        delta = newton_direction(y, p_n_device, v_n_device, m, g, dt)
        y = y + delta
        iterations[-1]["next_delta_norm"] = float(torch.linalg.vector_norm(delta).detach().cpu().item())
    return {"initial_y": [float(x) for x in initial_y.detach().cpu().tolist()], "iterations": iterations}

In [25]:
def plot_reference_residual_comparison(mlp_trajectory: dict, newton_trajectory: dict, save_path: Path, title: str) -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot([i["step"] for i in mlp_trajectory["iterations"]], [finite_plot_value(i["residual_norm"]) for i in mlp_trajectory["iterations"]], marker="o", label="MLP")
    ax.plot([i["step"] for i in newton_trajectory["iterations"]], [finite_plot_value(i["residual_norm"]) for i in newton_trajectory["iterations"]], marker="s", linestyle="--", label="Newton")
    ax.set_yscale("log"); ax.set_title(title); ax.set_xlabel("Iteration"); ax.set_ylabel(r"$\|\nabla E(y)\|_2$"); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.close(fig)


def plot_reference_trajectory_3d(mlp_trajectory: dict, newton_trajectory: dict, y_star: Sequence[float], save_path: Path, title: str) -> None:
    mlp_points = np.asarray([i["y"] for i in mlp_trajectory["iterations"]], dtype=float)
    newton_points = np.asarray([i["y"] for i in newton_trajectory["iterations"]], dtype=float)
    initial_point = np.asarray(mlp_trajectory["initial_y"], dtype=float)
    y_star_np = np.asarray(y_star, dtype=float)
    fig = plt.figure(figsize=(10, 8)); ax = fig.add_subplot(111, projection="3d")
    ax.plot(mlp_points[:,0], mlp_points[:,1], mlp_points[:,2], "-o", label="MLP")
    ax.plot(newton_points[:,0], newton_points[:,1], newton_points[:,2], "--s", label="Newton")
    ax.scatter(*initial_point, marker="x", s=140, linewidths=2.0, label="initial")
    ax.scatter(*y_star_np, marker="*", s=320, label=r"$y^*$")
    set_equal_3d_axes(ax, np.vstack([mlp_points, newton_points, initial_point[None,:], y_star_np[None,:]]))
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z"); ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.close(fig)


def plot_reference_energy_contour_2d(mlp_trajectory: dict, newton_trajectory: dict, y_star: Sequence[float], p_n: Sequence[float], v_n: Sequence[float],
                                     m: float, g: float, dt: float, save_path: Path, title: str) -> None:
    mlp_points = np.asarray([i["y"] for i in mlp_trajectory["iterations"]], dtype=float)
    newton_points = np.asarray([i["y"] for i in newton_trajectory["iterations"]], dtype=float)
    y_star_np = np.asarray(y_star, dtype=float); p_n_np = np.asarray(p_n, dtype=float); v_n_np = np.asarray(v_n, dtype=float)
    initial_np = np.asarray(mlp_trajectory["initial_y"], dtype=float)
    projected = np.vstack([mlp_points[:,[0,2]], newton_points[:,[0,2]], y_star_np[[0,2]][None,:], initial_np[[0,2]][None,:]])
    lower = projected.min(axis=0); upper = projected.max(axis=0); span = np.maximum(upper-lower, 2e-4)
    lower -= 0.2*span; upper += 0.2*span
    x_values = np.linspace(lower[0], upper[0], 240); z_values = np.linspace(lower[1], upper[1], 240)
    x_grid, z_grid = np.meshgrid(x_values, z_values)
    points = np.broadcast_to(y_star_np.reshape(1,1,3), (240,240,3)).copy(); points[...,0] = x_grid; points[...,2] = z_grid
    residual = points - p_n_np - dt * v_n_np
    energy = (m/(2.0*dt**2))*np.sum(residual**2, axis=-1) + m*g*points[...,2]
    y_star_residual = y_star_np - p_n_np - dt*v_n_np
    e_star = (m/(2.0*dt**2))*np.sum(y_star_residual**2) + m*g*y_star_np[2]
    gap = np.maximum(energy - e_star, PLOT_FLOOR)
    max_gap = float(np.max(gap)); min_level = max(max_gap*1e-8, PLOT_FLOOR); max_level = max(max_gap, min_level*10.0)
    levels = np.geomspace(min_level, max_level, 28)
    fig, ax = plt.subplots(figsize=(9,7))
    contour = ax.contourf(x_grid, z_grid, gap, levels=levels, norm=matplotlib.colors.LogNorm(vmin=min_level, vmax=max_level), alpha=0.82, extend="both")
    ax.plot(mlp_points[:,0], mlp_points[:,2], "-o", label="MLP")
    ax.plot(newton_points[:,0], newton_points[:,2], "--s", label="Newton")
    ax.scatter(initial_np[0], initial_np[2], marker="x", s=120, linewidths=2.0, label="initial")
    ax.scatter(y_star_np[0], y_star_np[2], marker="*", s=260, label=r"$y^*$")
    ax.set_xlabel("x"); ax.set_ylabel("z"); ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.25)
    fig.colorbar(contour, ax=ax).set_label(r"$E(y)-E(y^*)$")
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.close(fig)

In [26]:
def find_best_matching_experiment(experiment_files: Sequence[ExperimentFile], optimizer_name: str, learning_rate: float, target_dataset_size: int) -> ExperimentFile:
    candidates = []
    for exp in experiment_files:
        cfg = exp.report.get("config", {})
        if str(cfg.get("optimizer_name", "")).lower() != optimizer_name.lower():
            continue
        if not math.isclose(float(cfg.get("learning_rate", float("nan"))), learning_rate, rel_tol=0.0, abs_tol=learning_rate*1e-8):
            continue
        actual = int(cfg.get("actual_dataset_size", parse_dataset_size_from_name(exp.experiment_dir.name) or -1))
        candidates.append((abs(actual - target_dataset_size), actual, exp))
    if not candidates:
        raise FileNotFoundError(f"Cannot find optimizer={optimizer_name}, lr={learning_rate}, N≈{target_dataset_size}")
    candidates.sort(key=lambda x: (x[0], x[1]))
    return candidates[0][2]


def select_detailed_test_indices(num_test_points: int, num_points: int = 3) -> list[int]:
    if num_test_points < num_points:
        return list(range(num_test_points))
    return sorted(set(np.linspace(0, num_test_points - 1, num_points).round().astype(int).tolist()))


def run_detailed_test_point_plots():
    detailed_root = OUTPUT_DIR / "detailed_test_points"
    detailed_root.mkdir(parents=True, exist_ok=True)
    selected_test_indices = select_detailed_test_indices(int(test_points_cpu.shape[0]), NUM_DETAILED_TEST_POINTS)
    results = []
    for target_size in DETAILED_TARGET_DATASET_SIZES:
        exp = find_best_matching_experiment(experiment_files, DETAILED_OPTIMIZER_NAME, DETAILED_LEARNING_RATE, target_size)
        cfg = exp.report.get("config", {})
        actual_size = int(cfg.get("actual_dataset_size", parse_dataset_size_from_name(exp.experiment_dir.name) or -1))
        model, dtype, tensors = instantiate_model_from_report_and_state(experiment=exp, device=device)
        model_dir = detailed_root / f"{DETAILED_OPTIMIZER_NAME}_lr_{DETAILED_LEARNING_RATE:.0e}_N_{actual_size}"
        model_dir.mkdir(parents=True, exist_ok=True)
        for local_id, test_index in enumerate(selected_test_indices):
            initial_y = test_points_cpu[test_index].to(dtype=dtype)
            point_dir = model_dir / f"test_point_{local_id:03d}_index_{test_index}"
            point_dir.mkdir(parents=True, exist_ok=True)
            mlp_trajectory = evaluate_single_trajectory(model, initial_y, tensors["p_n"], tensors["v_n"], y_star_cpu.to(dtype=dtype), m, g, dt, STEPS, device, dtype)
            newton_trajectory = evaluate_newton_trajectory(initial_y, tensors["p_n"], tensors["v_n"], y_star_cpu.to(dtype=dtype), m, g, dt, STEPS, device, dtype)
            title_prefix = f"Adam lr=1e-4, N={actual_size:,}, test point {local_id} index {test_index}"
            plot_reference_residual_comparison(mlp_trajectory, newton_trajectory, point_dir / "final_reference_residual_comparison.png", title_prefix + "\nResidual comparison")
            plot_reference_energy_contour_2d(mlp_trajectory, newton_trajectory, y_star_cpu.tolist(), p_n_cpu.tolist(), v_n_cpu.tolist(), m, g, dt, point_dir / "final_reference_energy_contour_2d.png", title_prefix + "\nEnergy contour")
            plot_reference_trajectory_3d(mlp_trajectory, newton_trajectory, y_star_cpu.tolist(), point_dir / "final_reference_trajectory_3d.png", title_prefix + "\n3D trajectory")
            results.append({"actual_dataset_size": actual_size, "test_index": int(test_index), "output_dir": str(point_dir), "initial_y": initial_y.detach().cpu().tolist()})
    return results

if DO_DETAILED_TRAJECTORIES:
    detailed_results = run_detailed_test_point_plots()
    pd.DataFrame(detailed_results)
else:
    detailed_results = []

## 17. 保存 summary

保存所有测试结果、诊断结果和详细轨迹目录。


In [27]:
def make_json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {k: make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return make_json_safe(value.tolist())
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

summary = {
    "results_dir": str(RESULTS_DIR),
    "output_dir": str(OUTPUT_DIR),
    "device": str(device),
    "test_set": test_metadata,
    "physics": {"p_n": p_n_cpu.tolist(), "v_n": v_n_cpu.tolist(), "m": m, "g": g, "dt": dt, "y_star": y_star_cpu.tolist()},
    "evaluation": {
        "steps": int(STEPS),
        "batch_size": int(BATCH_SIZE),
        "pointwise_metrics": {
            "residual": r"r_k(y_0)=||∇E(y^(k)(y_0))||_2",
            "loss_gap": r"ell_k(y_0)=E(y^(k)(y_0))-E(y*)",
        },
    },
    "initial_distribution_diagnostic": initial_diag_df.to_dict(orient="records"),
    "pn_vs_test_mean_diagnostic": diag_df.to_dict(orient="records"),
    "identity_check_for_selected_model": identity_check_df.to_dict(orient="records"),
    "experiments": records,
    "detailed_test_point_results": detailed_results,
}
summary_path = OUTPUT_DIR / "heldout_test_residual_loss_notebook_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(summary), f, indent=2, ensure_ascii=False)
print("saved:", summary_path)
print("all outputs under:", OUTPUT_DIR)

saved: /data/zhoucy/sim_newton/unit_test_for_scale_data/second_stage_test/free_fall_regular_grid_fullbatch_50000_float64/heldout_test_residual_loss_notebook/heldout_test_residual_loss_notebook_summary.json
all outputs under: /data/zhoucy/sim_newton/unit_test_for_scale_data/second_stage_test/free_fall_regular_grid_fullbatch_50000_float64/heldout_test_residual_loss_notebook


## 18. 怎么判断是不是代码问题？

建议按下面顺序看：

1. `overlap_counts` 是否全是 0：如果不是，测试集剔除训练集有问题。
2. `p_n is inside y_star-centered cube` 是否为 True：如果是，说明 `p_n` 本来就是测试区域内的点。
3. `initial_diag_df` 里 `pn_percentile_in_test`：如果在 20% 到 80% 之间，说明 `p_n` 在测试分布里很普通。
4. `diagnostic_pn_vs_test_mean_relative_difference.png`：看相对差，而不是只看 log 总图。
5. `diagnostic_final_distribution_vs_pn.png`：看 `p_n` 处在最终测试误差分布中的哪个位置。
6. `identity_check_df`：如果 loss/residual 理论关系误差很小，说明 residual 和 loss 的计算本身基本可信。

如果这些检查都正常，那么“全测试集结果和 `p_n` 看起来接近”更可能是测试集设计导致的，而不是代码 bug。当前测试集是以 `y*` 为中心的小局部 cube，而 `p_n` 也在这个 cube 内，所以它们本来就不是强区分的两个测试对象。

如果你想更明显地区分测试集和 `p_n`，可以额外加：

- `shell test`：只采样 cube 外壳；
- `near-p_n test`：以 `p_n` 为中心采样；
- `larger-radius test`：半径改成训练半径的 2 倍或 5 倍；
- `line test`：沿着 `p_n -> y*` 或随机方向采样。
